# Constrained Batch Shift — A100 experiment

## 1. What this experiment tests

We are testing whether posterior-to-decision shift naturally becomes large near competitive constrained batch BO decisions and whether practical scrambled-Sobol qLogEI-style estimates then become unreliable. The experiment compares a matched Gaussian GP belief with a conjugate Student-t-process belief on eight prospectively frozen constrained Hartmann6 states.

No new inference method, sampler, or acquisition is being tested. This is a diagnostic falsification experiment. A partial run cannot determine the paper gate.

## 2. Frozen protocol

The next code cell reads and displays the authoritative repository configuration. It does not duplicate editable scientific constants. The frozen protocol contains the eight state seeds, q=4, the standard smooth Hartmann6 constraint, matched Gaussian/Student-t beliefs, candidate-panel construction, practical QMC counts, high-budget reference, and mechanical GO/NO-GO thresholds.

## 3. Colab setup instructions

1. Open **Runtime → Change runtime type**.
2. Select an **A100 GPU** if available.
3. Run cells from top to bottom.
4. First run the GPU preflight.
5. Only if the preflight prints `GPU PREFLIGHT: PASS`, enable and run the full experiment.
6. Leave the notebook running until the final result bundle is produced. Each completed state is saved immediately.
7. Download `constrained_batch_shift_gpu_results.zip`, place it in the root of your local repository, and give it to Codex using the separate GPU-results finalization prompt.

Google Drive is optional but strongly recommended because Colab local storage disappears on disconnect. The notebook never authenticates to or pushes to GitHub.

In [ ]:
# Human controls: change only these booleans, not the frozen scientific config.
REPOSITORY_URL = 'https://github.com/PaulsonLab/energy-inference-bo.git'
IMPLEMENTATION_REF = 'constrained-batch-shift-gpu-v1'
USE_DRIVE = True
RUN_PREFLIGHT = False
RUN_FULL = False


## 4. Environment setup

This cell installs the locked project into its own `.venv`, checks out the stable implementation tag, rejects a non-A100 runtime, and records the exact Git SHA and protocol hash. It does not require a runtime restart.

In [ ]:
import hashlib, json, os, pathlib, shutil, subprocess, sys
repo = pathlib.Path('/content/energy-inference-bo')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'fetch', '--tags', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', IMPLEMENTATION_REF], check=True)
sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
if shutil.which('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'sync', '--locked', '--group', 'dev'], cwd=repo, check=True)
project_python = repo / '.venv' / 'bin' / 'python'
scientific_env = os.environ.copy()
scientific_env['MPLBACKEND'] = 'Agg'
runtime_code = """import json, torch, botorch, gpytorch, scipy, numpy; print(json.dumps({'python': __import__('sys').version, 'torch': torch.__version__, 'botorch': botorch.__version__, 'gpytorch': gpytorch.__version__, 'scipy': scipy.__version__, 'numpy': numpy.__version__, 'cuda_available': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))"""
runtime = json.loads(subprocess.check_output([str(project_python), '-c', runtime_code], cwd=repo, env=scientific_env, text=True))
if not runtime['cuda_available']:
    raise RuntimeError('A CUDA GPU is required. Select Runtime → Change runtime type → A100 GPU.')
if 'A100' not in runtime['gpu']:
    raise RuntimeError(f"Full protocol requires an A100; active device is {runtime['gpu']}")
config_path = repo / 'experiments' / 'constrained_batch_shift' / 'config.json'
config = json.loads(config_path.read_text())
canonical = json.dumps(config, sort_keys=True, separators=(',', ':')).encode()
protocol_hash = hashlib.sha256(canonical).hexdigest()
print(f"[SETUP] GPU: {runtime['gpu']}")
print(f'[SETUP] Repository SHA: {sha}')
print(f'[SETUP] Implementation ref: {IMPLEMENTATION_REF}')
print(f'[SETUP] Protocol hash: {protocol_hash}')
print('[SETUP] Versions:', json.dumps(runtime, indent=2))
print('[SETUP] Environment ready.')

In [ ]:
# Display values directly from the frozen config.
display_protocol = {
    'protocol_version': config['protocol_version'],
    'state_seeds': config['states']['seeds'],
    'q': config['acquisition']['batch_size'],
    'constraint': config['problem']['constraint_violation'] + ' <= 0',
    'feasible_fraction': config['problem']['exact_feasible_fraction'],
    'beliefs': ['matched Gaussian', config['beliefs']['student_t']],
    'candidate_batches': config['candidate_batches'],
    'practical_sample_counts': config['practical_qmc']['sample_counts'],
    'reference': config['reference'],
    'gate': config['gate'],
}
print(json.dumps(display_protocol, indent=2))

## 5. Persistent output directory

When `USE_DRIVE=True`, authorize the standard Colab Drive mount. Results are stored under `MyDrive/energy-inference-bo/constrained_batch_shift/<tag>-<protocol-hash>/`. Otherwise results remain under `/content/` and will be lost when the runtime disconnects. Rerunning this cell and later run cells resumes compatible completed states.

In [ ]:
run_id = f'{IMPLEMENTATION_REF}-{protocol_hash[:12]}'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    output_dir = pathlib.Path('/content/drive/MyDrive/energy-inference-bo/constrained_batch_shift') / run_id
else:
    output_dir = pathlib.Path('/content/constrained_batch_shift') / run_id
    print('[WARNING] Drive disabled: all checkpoints disappear if this runtime disconnects.')
output_dir.mkdir(parents=True, exist_ok=True)
manifest = {'git_sha': sha, 'implementation_ref': IMPLEMENTATION_REF, 'protocol_hash': protocol_hash, 'runtime': runtime, 'output_dir': str(output_dir)}
manifest_path = output_dir / 'colab_manifest.json'
if manifest_path.exists():
    prior = json.loads(manifest_path.read_text())
    if prior['protocol_hash'] != protocol_hash or prior['git_sha'] != sha:
        raise RuntimeError('Existing output directory belongs to an incompatible protocol or implementation.')
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n')
print('[SETUP] Persistent output:', output_dir)

## 6. Tests and GPU preflight

Run the locked unit suite first. Then set `RUN_PREFLIGHT=True` in the controls cell and rerun the preflight cell. It uses one frozen seed with deliberately reduced numerical settings and cannot determine the scientific gate. The full cell checks for its protocol-matched pass marker.

In [ ]:
subprocess.run([str(project_python), '-m', 'pytest', '-q'], cwd=repo, env=scientific_env, check=True)
print('[TESTS] Unit suite passed.')

In [ ]:
if not RUN_PREFLIGHT:
    print('GPU preflight guard active. Set RUN_PREFLIGHT=True and rerun this cell.')
else:
    subprocess.run([str(project_python), 'experiments/constrained_batch_shift/run.py', 'preflight', '--device', 'cuda', '--config', str(config_path), '--output-dir', str(output_dir)], cwd=repo, env=scientific_env, check=True)

## 7. FULL EXPERIMENT — RUN ONLY AFTER PREFLIGHT PASSES

Set `RUN_FULL=True` only after the preceding cell prints `GPU PREFLIGHT: PASS`. This processes all eight states sequentially. A matching completed state is skipped; an incompatible checkpoint causes a hard failure. Compact descriptive summaries are printed after each state, but no incomplete subset is interpreted as the paper gate.

In [ ]:
if not RUN_FULL:
    print('Full-run guard active. Set RUN_FULL=True only after GPU PREFLIGHT: PASS.')
else:
    subprocess.run([str(project_python), 'experiments/constrained_batch_shift/run.py', 'full', '--device', 'cuda', '--config', str(config_path), '--output-dir', str(output_dir)], cwd=repo, env=scientific_env, check=True)
    aggregate = json.loads((output_dir / 'aggregate_summary.json').read_text())
    print('All-state descriptive aggregate:')
    for state in aggregate['states']:
        print(state['seed'], {belief: {'median_ess': state['beliefs'][belief]['top_decile_median_ess_fraction'], 'shift_positive': state['beliefs'][belief]['shift_positive'], 'material_failure': state['beliefs'][belief]['material_failure']} for belief in ('gaussian', 'student_t')})

## 8. Result bundle

After all eight states complete, this cell creates exactly one portable ZIP plus `SHA256SUM.txt`. It contains the frozen config, hashes, environment, raw per-state results, aggregate metrics, checkpoint manifest, and progress records—no disposable caches.

In [ ]:
archive = pathlib.Path('/content/constrained_batch_shift_gpu_results.zip')
if not (output_dir / 'aggregate_summary.json').exists():
    raise RuntimeError('All eight states and aggregate output must exist before packaging.')
subprocess.run([str(project_python), 'experiments/constrained_batch_shift/run.py', 'package', '--config', str(config_path), '--output-dir', str(output_dir), '--archive', str(archive)], cwd=repo, env=scientific_env, check=True)
checksum_path = archive.with_name('SHA256SUM.txt')
checksum = checksum_path.read_text().split()[0]
if USE_DRIVE:
    shutil.copy2(archive, output_dir / archive.name)
    shutil.copy2(checksum_path, output_dir / checksum_path.name)
print('=' * 60)
print('GPU RUN COMPLETE')
print('=' * 60)
print('All 8/8 frozen states completed.')
print(f'\nResult bundle:\n  {archive}')
print(f'\nSHA256:\n  {checksum}')
if USE_DRIVE: print(f'\nDrive copy:\n  {output_dir / archive.name}')
print('''\nNEXT STEP:
1. Download constrained_batch_shift_gpu_results.zip.
2. Put the ZIP in the root of your local energy-inference-bo repository.
3. Return to Codex and run the GPU-results finalization prompt.
4. Do NOT rerun or modify the frozen protocol.
''' + '=' * 60)

## 9. Interpretation

Do not interpret a smoke, preflight, or partial state subset. The all-state aggregate is still subject to repository-side integrity and numerical audit. A positive mechanical label does not authorize a sampler; a negative result is a valid reason to reconsider the paper direction.

## 10. Next authorized action

Download the ZIP and place it in the repository root. The next Codex call may audit and finalize only these frozen GPU results. No decision-adapted inference implementation is authorized.

In [ ]:
# One-click browser download after the result-bundle cell succeeds.
from google.colab import files
if not pathlib.Path('/content/constrained_batch_shift_gpu_results.zip').exists():
    raise RuntimeError('Result ZIP does not exist yet.')
files.download('/content/constrained_batch_shift_gpu_results.zip')